# FFHQ-Wrinkle evaluation

Notebook นี้เรียกใช้ logic จาก `wrinkle_prototype.py` เพื่อสำรวจผลโดยไม่คัดลอกสูตรคะแนน เมื่อมี segmentation model แล้วจึงเพิ่ม Dice, IoU, precision และ recall จาก predicted mask ใน notebook นี้

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'ai':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'ai'))

from wrinkle_prototype import PreprocessConfig, analyze, find_image

In [ ]:
DATA_ROOT = ROOT / 'storage/data/non_time_serie/ffhq_wrinkle'
LIMIT = 100
CONFIG = PreprocessConfig()

images_dir = DATA_ROOT / 'images1024x1024'
masks_dir = DATA_ROOT / 'manual_wrinkle_masks'
print(f'Dataset: {DATA_ROOT}')

In [ ]:
if not images_dir.is_dir() or not masks_dir.is_dir():
    raise FileNotFoundError('วาง dataset ไว้ใน images1024x1024/ และ manual_wrinkle_masks/')

pairs = [(find_image(images_dir, mask.stem), mask) for mask in sorted(masks_dir.glob('*.png'))]
pairs = [(image, mask) for image, mask in pairs if image is not None][:LIMIT]
assert pairs, 'ไม่พบ image/mask ที่มีชื่อไฟล์ตรงกัน'

rows = []
for image_path, mask_path in pairs:
    result = analyze(image_path, mask_path, CONFIG)
    rows.append({
        'image': image_path.name,
        'wrinkle_area_ratio': result['wrinkle_area_ratio'],
        'wrinkle_score': result['wrinkle_score'],
        'severity': result['severity'],
        'quality_flags': ', '.join(result['quality_flags']),
    })

results = pd.DataFrame(rows)
results.head()

In [ ]:
display(results.describe(include='all'))
display(results['severity'].value_counts().rename_axis('severity').to_frame('count'))
results['wrinkle_area_ratio'].plot.hist(bins=20, title='Wrinkle area ratio')
plt.xlabel('Positive mask pixels / evaluated pixels')
plt.show()